# LuxTTS — voice cloning in Colab

**Demo** — clone a voice from a short sample and synthesise speech in the browser (GPU runtime).

<a href="https://colab.research.google.com/github/47096/lux-tts/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Runtime → Change runtime type → GPU (T4)**


In [ ]:
# Setup
print("Cloning LuxTTS...")
!git clone https://github.com/ysharma3501/LuxTTS.git
%cd LuxTTS
!uv pip install -r requirements.txt
!pip install k2==1.24.4.dev20251030+cuda12.6.torch2.9.0 -f https://k2-fsa.github.io/k2/cuda.html --no-deps


In [ ]:
# Load model
from zipvoice.luxvoice import LuxTTS
import soundfile as sf
from IPython.display import Audio, display
from google.colab import files

lux_tts = LuxTTS("YatharthS/LuxTTS", device="cuda")  # or cpu
print("Model ready")


In [ ]:
# Upload reference audio (5-20s)
print("Choose a WAV/MP3...")
uploaded = files.upload()
ref_path = list(uploaded.keys())[0]
prompt_audio, sr = sf.read(ref_path)
print(ref_path, sr, "samples:", len(prompt_audio))


In [ ]:
# Settings + text
text = "What was the noise? Can you hear that? It was so loud."

rms = 0.01
t_shift = 0.9
num_steps = 4
speed = 1.0
return_smooth = False
ref_duration = 10000


In [ ]:
# Generate
encoded_prompt = lux_tts.encode_prompt(prompt_audio, duration=ref_duration, rms=rms)
final_wav = lux_tts.generate_speech(
    text,
    encoded_prompt,
    num_steps=num_steps,
    t_shift=t_shift,
    speed=speed,
    return_smooth=return_smooth,
)
final_wav = final_wav.numpy().squeeze()
sf.write("output.wav", final_wav, 48000)
print("Saved output.wav")


In [ ]:
# Play + download
display(Audio(final_wav, rate=48000))
files.download("output.wav")
